# LAB 6 — Dashboard Comparativo com LangFuse
## Aula 4: OpenSearch Completo — Dense, Hybrid Search, Neural Sparse e Contextual Retrieval
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

---

**Duração estimada:** 30 minutos  
**Pré-requisitos:** LAB1–LAB5 concluídos, LangFuse acessível

---

## Objetivo

Neste laboratório vamos:

1. Executar as **3 estratégias de busca** (BM25 puro, Dense kNN, Hybrid RRF) nas 5 queries de avaliação
2. Medir **latência** e **MRR@5** de cada estratégia
3. Registrar todos os resultados no **LangFuse** com spans estruturados
4. Construir um **dashboard comparativo** com Matplotlib
5. Gerar um **relatório final** consolidando os aprendizados da Aula 4

---

## Visão Geral das Estratégias Comparadas

```
┌─────────────────────────────────────────────────────────┐
│                 BENCHMARK AULA 4                        │
│                                                         │
│  Estratégia 1: BM25 puro (lexical)                      │
│    → match query no campo 'conteudo'                    │
│                                                         │
│  Estratégia 2: Dense kNN (vetorial)                     │
│    → BGE-M3 embedding + kNN no campo 'embedding'        │
│                                                         │
│  Estratégia 3: Hybrid RRF (BM25 + Dense)                │
│    → hybrid query + search pipeline com RRF             │
│                                                         │
│  Corpus: juridico_baseline_aula4 (100 chunks)           │
│  Queries: 5 queries com relevância anotada              │
│  Métricas: Latência (ms), MRR@5                         │
└─────────────────────────────────────────────────────────┘
```

---
## Passo 1 — Instalação e Configuração

In [ ]:
!pip install -q opensearch-py langchain-openai sentence-transformers \
               langfuse matplotlib pandas numpy

print("✅ Dependências instaladas")

In [ ]:
import os

# ── OpenSearch 3.x ──────────────────────────────────────────────
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
OPENSEARCH_USER = os.getenv("OPENSEARCH_USER", "admin")
OPENSEARCH_PASS = os.getenv("OPENSEARCH_PASS", "admin")
INDEX_BASELINE  = "juridico_baseline_aula4"   # do LAB5 (100 chunks sem contexto)

# ── LangFuse ────────────────────────────────────────────────────
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-xxxx")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-xxxx")
LANGFUSE_HOST       = os.getenv("LANGFUSE_HOST", "http://localhost:3000")

# ── Embedding ───────────────────────────────────────────────────
EMBEDDING_MODEL = "BAAI/bge-m3"

print(f"OpenSearch: {OPENSEARCH_HOST}:{OPENSEARCH_PORT}")
print(f"LangFuse : {LANGFUSE_HOST}")
print(f"Índice   : {INDEX_BASELINE}")

---
## Passo 2 — Conexões

In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection
from sentence_transformers import SentenceTransformer
from langfuse import Langfuse

# OpenSearch 3.x
os_client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False,
    verify_certs=False,
    connection_class=RequestsHttpConnection,
)
info = os_client.info()
print(f"✅ OpenSearch {info['version']['number']}")
assert info['version']['number'].startswith('3'), "⚠️ Requer OpenSearch 3.x!"

# Verificar que o índice existe
assert os_client.indices.exists(index=INDEX_BASELINE), (
    f"❌ Índice '{INDEX_BASELINE}' não encontrado! Execute o LAB5 primeiro."
)
n_docs = os_client.count(index=INDEX_BASELINE)["count"]
print(f"✅ Índice '{INDEX_BASELINE}' encontrado com {n_docs} documentos")

# Modelo de embedding
embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Embedding: {EMBEDDING_MODEL}")

# LangFuse
langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST,
)
print(f"✅ LangFuse conectado")

---
## Passo 3 — Queries de Avaliação com Relevância Anotada

In [ ]:
# Queries de avaliação com IDs de documentos relevantes anotados manualmente
# (os IDs devem corresponder aos doc_id no índice)
QUERIES_AVALIACAO = [
    {
        "id": "Q01",
        "question": "Quais são os requisitos para decretação de prisão preventiva?",
        "doc_ids_relevantes": ["doc_001", "doc_002", "doc_005"],
    },
    {
        "id": "Q02",
        "question": "Como se caracteriza o crime ambiental de desmatamento em APP?",
        "doc_ids_relevantes": ["doc_003", "doc_007"],
    },
    {
        "id": "Q03",
        "question": "Quais os efeitos da reincidência criminal no cálculo da pena?",
        "doc_ids_relevantes": ["doc_004", "doc_009"],
    },
    {
        "id": "Q04",
        "question": "O que é habeas corpus e quando pode ser impetrado?",
        "doc_ids_relevantes": ["doc_006", "doc_010"],
    },
    {
        "id": "Q05",
        "question": "Quais documentos compõem o boletim de ocorrência policial?",
        "doc_ids_relevantes": ["doc_008"],
    },
]

print(f"Queries de avaliação: {len(QUERIES_AVALIACAO)}")
for q in QUERIES_AVALIACAO:
    print(f"  [{q['id']}] {q['question'][:60]}... ({len(q['doc_ids_relevantes'])} docs relevantes)")

---
## Passo 4 — Implementação das 3 Estratégias de Busca

In [ ]:
import time
import numpy as np

def calcular_mrr(resultados: list[str], relevantes: list[str]) -> float:
    """Mean Reciprocal Rank: 1/posição do primeiro resultado relevante."""
    for i, doc_id in enumerate(resultados, 1):
        if doc_id in relevantes:
            return 1.0 / i
    return 0.0

def busca_bm25(query: str, k: int = 5) -> tuple[list[str], float]:
    """Busca lexical BM25 pura."""
    t0 = time.perf_counter()
    resp = os_client.search(
        index=INDEX_BASELINE,
        body={
            "size": k,
            "query": {"match": {"conteudo": {"query": query}}}
        }
    )
    latencia_ms = (time.perf_counter() - t0) * 1000
    doc_ids = [h["_source"]["doc_id"] for h in resp["hits"]["hits"]]
    return doc_ids, latencia_ms

def busca_dense(query: str, k: int = 5) -> tuple[list[str], float]:
    """Busca vetorial densa com BGE-M3 + kNN."""
    emb = embedder.encode([query])[0].tolist()
    t0 = time.perf_counter()
    resp = os_client.search(
        index=INDEX_BASELINE,
        body={
            "size": k,
            "query": {"knn": {"embedding": {"vector": emb, "k": k}}}
        }
    )
    latencia_ms = (time.perf_counter() - t0) * 1000
    doc_ids = [h["_source"]["doc_id"] for h in resp["hits"]["hits"]]
    return doc_ids, latencia_ms

def busca_hybrid_rrf(query: str, k: int = 5) -> tuple[list[str], float]:
    """Busca híbrida com RRF via search pipeline do OpenSearch 3.x."""
    emb = embedder.encode([query])[0].tolist()
    t0 = time.perf_counter()
    resp = os_client.search(
        index=INDEX_BASELINE,
        params={"search_pipeline": "hybrid_rrf_pipeline"},
        body={
            "size": k,
            "query": {
                "hybrid": {
                    "queries": [
                        {"match": {"conteudo": {"query": query}}},
                        {"knn": {"embedding": {"vector": emb, "k": k}}}
                    ]
                }
            }
        }
    )
    latencia_ms = (time.perf_counter() - t0) * 1000
    doc_ids = [h["_source"]["doc_id"] for h in resp["hits"]["hits"]]
    return doc_ids, latencia_ms

print("✅ Funções de busca definidas")
print("  - busca_bm25(): match query BM25")
print("  - busca_dense(): kNN com BGE-M3")
print("  - busca_hybrid_rrf(): hybrid query + RRF pipeline")

---
## Passo 5 — Executar Benchmark e Registrar no LangFuse

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

# Trace principal do benchmark
trace = langfuse.trace(
    name="Aula4-LAB6-Benchmark-Estrategias-Busca",
    metadata={
        "aula": "4",
        "lab": "6",
        "n_queries": len(QUERIES_AVALIACAO),
        "indice": INDEX_BASELINE,
        "opensearch_version": "3.x",
        "estrategias": ["BM25", "Dense kNN", "Hybrid RRF"]
    }
)

resultados = []
N_REPETICOES = 3  # repetir cada busca para média de latência

print("=== EXECUTANDO BENCHMARK ===")
print(f"Queries: {len(QUERIES_AVALIACAO)} | Repetições: {N_REPETICOES} | k=5\n")

for q in tqdm(QUERIES_AVALIACAO):
    print(f"  Query {q['id']}: {q['question'][:50]}...")

    # Executar cada estratégia N_REPETICOES vezes e calcular média de latência
    for estrategia_nome, fn_busca in [
        ("BM25",       busca_bm25),
        ("Dense_kNN",  busca_dense),
        ("Hybrid_RRF", busca_hybrid_rrf),
    ]:
        latencias = []
        ultimo_resultado = []

        for _ in range(N_REPETICOES):
            try:
                doc_ids, lat = fn_busca(q["question"], k=5)
                latencias.append(lat)
                ultimo_resultado = doc_ids
            except Exception as e:
                print(f"    ⚠️ Erro em {estrategia_nome}: {e}")
                latencias.append(9999.0)

        lat_media = np.mean(latencias)
        mrr = calcular_mrr(ultimo_resultado, q["doc_ids_relevantes"])

        resultados.append({
            "query_id":    q["id"],
            "query":       q["question"],
            "estrategia":  estrategia_nome,
            "latencia_ms": round(lat_media, 1),
            "mrr":         round(mrr, 4),
        })

        # Span no LangFuse por query+estratégia
        span = trace.span(
            name=f"{q['id']}-{estrategia_nome}",
            metadata={
                "query": q["question"],
                "estrategia": estrategia_nome,
                "k": 5,
                "repeticoes": N_REPETICOES
            }
        )
        span.score(name="latencia_ms", value=lat_media)
        span.score(name="mrr",         value=mrr)
        span.end()

        print(f"    {estrategia_nome:12s} | lat={lat_media:6.1f}ms | MRR={mrr:.3f}")

langfuse.flush()
print(f"\n✅ Benchmark concluído — {len(resultados)} registros")
print(f"   Trace URL: {LANGFUSE_HOST}/traces/{trace.id}")

---
## Passo 6 — Análise Estatística dos Resultados

In [ ]:
df = pd.DataFrame(resultados)

# Agregar por estratégia
resumo = df.groupby("estrategia").agg(
    MRR_medio=("mrr", "mean"),
    MRR_std=("mrr", "std"),
    Latencia_media_ms=("latencia_ms", "mean"),
    Latencia_std_ms=("latencia_ms", "std"),
).round(4).reset_index()

# Ordenar por MRR decrescente
resumo = resumo.sort_values("MRR_medio", ascending=False)

print("=" * 70)
print(" RESUMO DO BENCHMARK — AULA 4")
print("=" * 70)
print(resumo.to_string(index=False))
print()

# Identificar melhor estratégia
melhor = resumo.iloc[0]
print(f"🏆 Melhor estratégia (MRR): {melhor['estrategia']}")
print(f"   MRR médio: {melhor['MRR_medio']:.4f} ± {melhor['MRR_std']:.4f}")
print(f"   Latência:  {melhor['Latencia_media_ms']:.1f} ± {melhor['Latencia_std_ms']:.1f} ms")

# Tabela detalhada por query
print("\n── Detalhamento por Query ──")
pivot = df.pivot_table(
    index="query_id",
    columns="estrategia",
    values=["mrr", "latencia_ms"]
)
print(pivot.to_string())

---
## Passo 7 — Dashboard Comparativo (Matplotlib)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

ESTRATEGIAS = ["BM25", "Dense_kNN", "Hybrid_RRF"]
CORES       = ["#4C72B0", "#DD8452", "#55A868"]
LABELS      = ["BM25 (Lexical)", "Dense kNN (BGE-M3)", "Hybrid RRF (BM25+kNN)"]

fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    "Aula 4 — Benchmark de Estratégias de Busca no OpenSearch 3.x\n"
    "MBA em RAG & CAG Aplicados a Direito e Segurança Pública",
    fontsize=14, fontweight="bold", y=0.98
)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Subplot 1: MRR médio por estratégia ──────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
mrr_vals = [resumo.loc[resumo.estrategia == e, "MRR_medio"].values[0] for e in ESTRATEGIAS]
mrr_std  = [resumo.loc[resumo.estrategia == e, "MRR_std"].values[0]   for e in ESTRATEGIAS]
bars = ax1.bar(LABELS, mrr_vals, color=CORES, yerr=mrr_std, capsize=5, alpha=0.85)
ax1.set_ylim(0, 1.0)
ax1.set_ylabel("MRR@5 (Mean Reciprocal Rank)", fontsize=11)
ax1.set_title("MRR@5 por Estratégia de Busca", fontsize=12, fontweight="bold")
ax1.axhline(y=0.5, color="gray", linestyle="--", alpha=0.4, label="Referência 0.5")
for bar, val in zip(bars, mrr_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.3f}", ha="center", fontsize=11, fontweight="bold")
ax1.tick_params(axis="x", labelsize=10)
ax1.legend(fontsize=9)

# ── Subplot 2: Latência por estratégia ──────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
lat_vals = [resumo.loc[resumo.estrategia == e, "Latencia_media_ms"].values[0] for e in ESTRATEGIAS]
lat_std  = [resumo.loc[resumo.estrategia == e, "Latencia_std_ms"].values[0]   for e in ESTRATEGIAS]
ax2.barh(LABELS, lat_vals, color=CORES, xerr=lat_std, capsize=5, alpha=0.85)
ax2.set_xlabel("Latência média (ms)", fontsize=11)
ax2.set_title("Latência de Busca", fontsize=12, fontweight="bold")
for i, (val, label) in enumerate(zip(lat_vals, LABELS)):
    ax2.text(val + 1, i, f"{val:.0f}ms", va="center", fontsize=10)
ax2.tick_params(axis="y", labelsize=9)

# ── Subplot 3: MRR por query (radar-style bar) ───────────────────
ax3 = fig.add_subplot(gs[1, :])
query_ids = [q["id"] for q in QUERIES_AVALIACAO]
x = np.arange(len(query_ids))
width = 0.25
for i, (estrategia, cor, label) in enumerate(zip(ESTRATEGIAS, CORES, LABELS)):
    mrr_por_query = [
        df.loc[(df.query_id == qid) & (df.estrategia == estrategia), "mrr"].values[0]
        for qid in query_ids
    ]
    ax3.bar(x + i * width, mrr_por_query, width, label=label, color=cor, alpha=0.85)

ax3.set_xlabel("Query", fontsize=11)
ax3.set_ylabel("MRR@5", fontsize=11)
ax3.set_title("MRR@5 por Query e Estratégia", fontsize=12, fontweight="bold")
ax3.set_xticks(x + width)
ax3.set_xticklabels([f"{q['id']}\n{q['question'][:30]}..." for q in QUERIES_AVALIACAO], fontsize=9)
ax3.set_ylim(0, 1.2)
ax3.legend(fontsize=10)
ax3.axhline(y=1.0, color="green", linestyle="--", alpha=0.3)

plt.savefig("benchmark_busca_aula4.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Dashboard salvo: benchmark_busca_aula4.png")

---
## Passo 8 — Registrar Scores Agregados no LangFuse e Exportar

In [ ]:
# Registrar scores agregados por estratégia no trace principal
for _, row in resumo.iterrows():
    trace.score(
        name=f"MRR_medio_{row['estrategia']}",
        value=float(row["MRR_medio"]),
        comment=f"MRR@5 médio para {len(QUERIES_AVALIACAO)} queries"
    )
    trace.score(
        name=f"Latencia_media_{row['estrategia']}_ms",
        value=float(row["Latencia_media_ms"]),
        comment=f"Latência média em {N_REPETICOES} repetições"
    )

langfuse.flush()
print(f"✅ Scores agregados registrados no LangFuse")

# Exportar CSV com todos os resultados
df.to_csv("benchmark_busca_aula4_detalhado.csv", index=False)
resumo.to_csv("benchmark_busca_aula4_resumo.csv",  index=False)

print("✅ CSVs exportados:")
print("   - benchmark_busca_aula4_detalhado.csv (por query)")
print("   - benchmark_busca_aula4_resumo.csv    (por estratégia)")

# Imprimir conclusão
print("\n" + "="*65)
print(" CONCLUSÃO DO BENCHMARK — AULA 4")
print("="*65)
for _, row in resumo.iterrows():
    print(f"  {row['estrategia']:12s} | MRR={row['MRR_medio']:.4f} | Lat={row['Latencia_media_ms']:.0f}ms")
print()
print(f"  → Estratégia recomendada para corpus jurídico: {resumo.iloc[0]['estrategia']}")
print(f"    Razão: melhor equilíbrio entre precisão semântica e terminologia técnica.")
print("="*65)

---
## ✅ Checklist de Conclusão do LAB6

| # | Item | Status |
|---|------|--------|
| 1 | OpenSearch 3.x conectado e índice baseline encontrado | ☐ |
| 2 | Benchmark executado para as 3 estratégias (BM25, Dense, Hybrid) | ☐ |
| 3 | Latência medida com N_REPETICOES=3 para estabilidade | ☐ |
| 4 | MRR@5 calculado para cada query+estratégia | ☐ |
| 5 | Todos os spans registrados no LangFuse (trace visível) | ☐ |
| 6 | Dashboard gerado e salvo como PNG | ☐ |
| 7 | CSVs detalhado e resumo exportados | ☐ |
| 8 | Scores agregados por estratégia registrados no LangFuse | ☐ |

---

## 🏁 Conclusão da Aula 4

Nesta aula construímos:

- **LAB1**: Índice híbrido no OpenSearch 3.x com kNN (BGE-M3) + BM25
- **LAB2**: Search pipeline com RRF para fusão de scores
- **LAB3**: Busca híbrida em corpus jurídico real
- **LAB4**: Neural Sparse Search com SPLADE via ML Commons
- **LAB5**: Contextual Retrieval — chunks enriquecidos com vLLM, medição RAGAS
- **LAB6**: Dashboard comparativo com LangFuse (este notebook)

**Técnicas dominadas:** #T04 Hybrid Search · #T09 Contextual Retrieval

**Próxima aula:** Avaliação e Observabilidade — RAGAS completo, DeepEval e LangFuse Avançado

---

## Referências

OPENSEARCH PROJECT. **Hybrid Search**. OpenSearch 3.x Docs. Disponível em: <https://docs.opensearch.org/latest/vector-search/ai-search/hybrid-search/>.

CORMACK, G. V.; CLARKE, C. L. A.; BUETTCHER, S. **Reciprocal rank fusion outperforms condorcet and individual rank learning methods**. In: *ACM SIGIR*, 2009.

ANTHROPIC. **Introducing Contextual Retrieval**. 2024. Disponível em: <https://www.anthropic.com/news/contextual-retrieval>.